## Nationality / Nacionalidades

In [ ]:
from timelink.notebooks import TimelinkNotebook

tlnb = TimelinkNotebook()
tlnb.print_info(show_token=True)

In [ ]:
import pandas as pd
from timelink.pandas import attribute_values
# set max rows to 500
pd.set_option('display.max_rows', 500)

df_totals = attribute_values('nacionalidade',db=tlnb.db)
df_totals

### List of all with attribute "nacionalidade"

In [ ]:
%pip install pyuca openpyxl

In [ ]:
import pyuca  # to sort accented characters properly

import pandas as pd
from timelink.pandas import entities_with_attribute
# show 500 rows
pd.set_option('display.max_rows', 550)

nacionais = entities_with_attribute(
    entity_type='person',
    show_elements=['id','name','groupname'],
    the_type='nacionalidade',
    more_attributes=['nascimento'],
    db=tlnb.db,
)
# filter groupname = 'n' (avoid "referido", "pai", "mãe")
nacionais = nacionais[nacionais.groupname=='n']
nacionais.info()

### Export all with attribute "nacionalidade"

In [ ]:
nacionais.to_excel("../inferences/paises_pessoas_n.xlsx", sheet_name='Sheet1', index=False)

### Group by nacionalidade

In [ ]:
paises_totais=nacionais.groupby('nacionalidade')['id_1'].nunique().reset_index().sort_values('id_1',ascending=False)
paises_totais

In [ ]:
paises_totais.to_excel("../inferences/paises_totais_n.xlsx", sheet_name='Sheet1', index=False)

### Countries and evolution of departures

In [ ]:
import pyuca  # to sort accented characters properly

import pandas as pd
from timelink.pandas import entities_with_attribute
# show 500 rows
pd.set_option('display.max_rows', 550)

embarques = entities_with_attribute(
    entity_type='person',
    show_elements=['id','name','groupname'],
    the_type='nacionalidade',
    more_attributes=['wicky'],
    db=tlnb.db,
)
# filter groupname = 'n' (avoid "referido", "pai", "mãe")
embarques = embarques[embarques.groupname=='n']
embarques['wicky.date'] = embarques['wicky.date'].fillna('0000')
embarques.info()
cols =['name','nacionalidade','wicky.date']
# replace NaN in wicky.date with 0000

embarques[cols].sort_values('wicky.date')

#### Group by country decade

In [ ]:
import pandas as pd
from timelink.kleio.utilities import convert_timelink_date

# extract the year from wicky.date
embarques['data_embarque'] = embarques['wicky.date'].apply(convert_timelink_date)
embarques['data_embarque'] = embarques['data_embarque'].fillna('0000-01-01')
embarques['ano_embarque'] = pd.to_datetime(embarques['data_embarque'], format='%Y-%m-%d', errors='coerce').dt.year
# count ano_entrada per periods of 10 years starting from 1540
embarques['periodo_embarque'] = embarques['ano_embarque'] // 10 * 10
# show columns jesuita-entrada.date, ano_entrada, periodo
embarques[['wicky.date','ano_embarque','periodo_embarque']].sort_values('wicky.date').tail(10)
# count number of embarques per period
embarques.groupby(['nacionalidade','periodo_embarque',])['id_1'].count().reset_index().sort_values(['nacionalidade','periodo_embarque',])


### Countries and place of entry / Países e locais de entrada

In [ ]:

import pyuca  # to sort accented characters properly

import pandas as pd
from timelink.pandas import entities_with_attribute
# show 500 rows
pd.set_option('display.max_rows', 550)

nacionalidade_entrada = entities_with_attribute(
    entity_type='person',
    show_elements=['id','name','groupname'],
    the_type='nacionalidade',
    more_attributes=['nascimento', 'jesuita-entrada','jesuita-entrada-x'],
    db=tlnb.db,
)
# filter groupname = 'n' (avoid "referido", "pai", "mãe")
nacionalidade_entrada = nacionalidade_entrada[nacionalidade_entrada.groupname=='n']
nacionalidade_entrada['jesuita-entrada'] = nacionalidade_entrada['jesuita-entrada'].fillna('?')
nacionalidade_entrada.info()

In [ ]:
nacionalidade_entrada.groupby(['nacionalidade','jesuita-entrada'])['id_1'].nunique()
df = nacionalidade_entrada.groupby(['nacionalidade','jesuita-entrada'])['id_1'].nunique().reset_index()
df_subtotal = df.groupby('nacionalidade')['id_1'].sum().reset_index()
df_subtotal.columns = ['nacionalidade', 'subtotal']
df = pd.merge(df, df_subtotal, on='nacionalidade')
df['perc_nacionalidade'] = df['id_1']/df['subtotal']
df.sort_values(['subtotal','nacionalidade','id_1'], ascending=[False,True,False])


In [ ]:
df = nacionalidade_entrada.groupby(['nacionalidade','jesuita-entrada'])['id_1'].nunique().reset_index()
df_subtotal = df.groupby('nacionalidade')['id_1'].sum().reset_index()
df_subtotal.columns = ['nacionalidade', 'subtotal']
df = pd.merge(df, df_subtotal, on='nacionalidade')
df['perc_nacionalidade'] = df['id_1']/df['subtotal']
df